This lecture introduces GPU computing in Julia.

## GPGPU

GPUs are ubiquitous in modern computers. Following are NVIDIA GPUs on today's typical computer systems.

| NVIDIA GPUs         | H100 PCIe                           | RTX 6000                                 | RTX 5000                              |
|---------------------|----------------------------------------|-----------------------------------------|--------------------------------------|
|                     | ![H100](nvidia_h100.png) | ![RTX 6000](nvidia_rtx6000.png)    | ![RTX 5000](nvidia_rtx5000.png) |
| Computers           | servers, cluster                       | desktop                                 | laptop                               |
|                     | ![Server](gpu_server.jpg)       | ![Desktop](alienware-area51.png) | ![Laptop](macpro_inside.png)  |
| Main usage          | scientific computing                   | daily work, gaming                      | daily work                           |
| Memory              | 80 GB                                    | 48 GB                                   | 16 GB                                  |
| Memory bandwidth    | 2 TB/sec                              | 960 GB/sec                               | 576 GB/sec                             |
| Number of cores     | ???                                    | ???                                     | ???                                  |
| Processor clock     | ??? GHz                                 | ??? GHz                                  | ??? GHz                               |
| Peak DP performance | 26 TFLOPS                              | ??? TFLOPS                                        |                                    ??? TFLOPS  |
| Peak SP performance | 51 TFLOPS                            | 91.1 TFLOPS                              | 42.6 TFLOPS                            |

## GPU architecture vs CPU architecture

* GPUs contain 1000s of processing cores on a single card; several cards can fit in a desktop PC  

* Each core carries out the same operations in parallel on different input data -- single program, multiple data (SPMD) paradigm  

* Extremely high arithmetic intensity *if* one can transfer the data onto and results off of the processors quickly

| ![i7 die](cpu_i7_die.png) | ![Fermi die](Fermi_Die.png) |
|----------------------------------|------------------------------------|
| ![Einstein](einstein.png) | ![Rain man](rainman.png)    |

## GPGPU in Julia

GPU support by Julia is under active development. Check [JuliaGPU](https://github.com/JuliaGPU) for currently available packages. 

There are multiple paradigms to program GPU in Julia, depending on the specific hardware.

- **CUDA** is an ecosystem exclusively for Nvidia GPUs. There are extensive CUDA libraries for scientific computing: CuBLAS, CuRAND, CuSparse, CuSolve, CuDNN, ...

  The [CUDA.jl](https://github.com/JuliaGPU/CUDA.jl) package allows defining arrays on **Nvidia GPUs** and overloads many common operations.

- The [AMDGPU.jl](https://github.com/JuliaGPU/AMDGPU.jl) package allows defining arrays on **AMD GPUs** and overloads many common operations.

- The [Metal.jl](https://github.com/JuliaGPU/Metal.jl) package allows defining arrays on **Apple Silicon** GPU and overloads many common operations.  

    [AppleAccelerate.jl](https://github.com/JuliaLinearAlgebra/AppleAccelerate.jl) wraps the [macOS Accelerate framework](https://developer.apple.com/documentation/accelerate), which provides high-performance libraries for linear algebra, signal processing, and image processing on Apple Silicon CPU. This is analog of MKL for Intel CPU.

- The [oneAPI.jl](https://github.com/JuliaGPU/oneAPI.jl) package allows defining arrays on **Intel GPUs** and overloads many common operations.

I'll illustrate using Metal.jl on my MacBook Pro running MacOS Sequoia 15.4. It has Apple M2 chip with 38 GPU cores.

In [28]:
versioninfo()

Julia Version 1.12.6
Commit 15346901f00 (2026-04-09 19:20 UTC)
Build Info:
  Official https://julialang.org release
Platform Info:
  OS: macOS (arm64-apple-darwin24.0.0)
  CPU: 12 × Apple M2 Max
  WORD_SIZE: 64
  LLVM: libLLVM-18.1.7 (ORCJIT, apple-m2)
  GC: Built with stock GC
Threads: 8 default, 1 interactive, 8 GC (on 8 virtual cores)
Environment:
  JULIA_NUM_THREADS = 8
  JULIA_EDITOR = code


Load packages:

In [29]:
using Pkg

Pkg.activate(pwd())
Pkg.instantiate()
Pkg.status()

  Activating project at `~/Documents/github.com/ucla-biostat-257/2026spring/slides/09-juliagpu`


Status `~/Documents/github.com/ucla-biostat-257/2026spring/slides/09-juliagpu/Project.toml`
  [6e4b80f9] BenchmarkTools v1.7.0
  [bdcacae8] LoopVectorization v0.12.173
  [dde4c033] Metal v1.9.3
  [37e2e46d] LinearAlgebra v1.12.0


## Query GPU devices in the system

In [30]:
using Metal

Metal.versioninfo()

macOS 26.4.0, Darwin 25.4.0

Toolchain:
- Julia: 1.12.6
- LLVM: 18.1.7

Julia packages:
- Metal.jl: 1.9.3
- GPUArrays: 11.4.1
- GPUCompiler: 1.9.1
- KernelAbstractions: 0.9.41
- ObjectiveC: 3.4.2
- LLVM: 9.4.6
- LLVMDowngrader_jll: 0.6.0+1

1 device:
- Apple M2 Max 38 GPU cores (50.001 GiB allocated)


## Transfer data between main memory and GPU

In [31]:
using Random
Random.seed!(257)

# generate SP data on CPU
x = rand(Float32, 3, 3)
# transfer data form CPU to GPU
xd = MtlArray(x)

3×3 MtlMatrix{Float32, Metal.PrivateStorage}:
 0.145793  0.939801  0.479926
 0.567772  0.577251  0.81655
 0.800538  0.38893   0.914135

In [32]:
# generate array on GPU directly
# yd = Metal.ones(3, 3)
yd = MtlArray(ones(Float32, 3, 3))

3×3 MtlMatrix{Float32, Metal.PrivateStorage}:
 1.0  1.0  1.0
 1.0  1.0  1.0
 1.0  1.0  1.0

In [33]:
# collect data from GPU to CPU
x = collect(xd)

3×3 Matrix{Float32}:
 0.145793  0.939801  0.479926
 0.567772  0.577251  0.81655
 0.800538  0.38893   0.914135

## Linear algebra

In [34]:
using BenchmarkTools, LinearAlgebra, Random

Random.seed!(257)

n = 2^14
# on CPU
x = rand(Float32, n, n)
y = rand(Float32, n, n)
z = zeros(Float32, n, n)
# on GPU
xd = MtlArray(x)
yd = MtlArray(y)
zd = MtlArray(z);

### Dot product

In [35]:
# SP matrix dot product on CPU: tr(X'Y)
bm_cpu = @benchmark dot($x, $y)

BenchmarkTools.Trial: 276 samples with 1 evaluation per sample.
 Range (min … max):  17.546 ms … 31.910 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     17.922 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):   18.126 ms ±  1.185 ms  ┊ GC (mean ± σ):  0.00% ± 0.00%

    ▄▅▂█▅▂▅                                                    
  ▄█████████▇▅▅▁▄▄▅▃▃▃▃▂▃▁▃▂▁▁▁▃▂▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▂ ▃
  17.5 ms         Histogram: frequency by time        20.9 ms <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [36]:
# SP matrix dot product on GPU: tr(X'Y)
# why are there allocations?
bm_gpu = @benchmark Metal.@sync dot($xd, $yd)

BenchmarkTools.Trial: 637 samples with 1 evaluation per sample.
 Range (min … max):  7.544 ms …   9.718 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     7.838 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   7.847 ms ± 229.534 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

    ▂  ▄█▁            ▁                                        
  ▄▆█▆▅███▇▆▄▅▅▅▄▅▇█▆▆██▇▇▆▅▇▆▄▅▅▃▃▃▃▃▃▃▃▁▂▁▂▂▂▃▁▂▁▁▁▂▂▁▂▁▂▂▂ ▃
  7.54 ms         Histogram: frequency by time        8.61 ms <

 Memory estimate: 16.77 KiB, allocs estimate: 538.

In [37]:
# speedup on GPU over CPU
median(bm_cpu.times) / median(bm_gpu.times)

2.2865315131411075

### Broadcast

In [38]:
# SP broadcast on CPU: z .= x .* y
bm_cpu = @benchmark $z .= $x .* $y

BenchmarkTools.Trial: 138 samples with 1 evaluation per sample.
 Range (min … max):  35.088 ms … 41.965 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     35.948 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):   36.378 ms ±  1.294 ms  ┊ GC (mean ± σ):  0.00% ± 0.00%

      █▄▁▁     ▁                                               
  ▃▄▅▅████▅▅▇▆██▃▃▃▃▃▁▁▃▃▁▃▁▁▁▃▃▃▃▁▃▃▁▁▁▁▃▁▃▁▁▁▁▁▁▁▁▃▁▁▁▁▃▁▁▃ ▃
  35.1 ms         Histogram: frequency by time        41.5 ms <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [39]:
# SP broadcast on GPU: z .= x .* y
# why is there allocation?
bm_gpu = @benchmark Metal.@sync $zd .= $xd .* $yd

BenchmarkTools.Trial: 549 samples with 1 evaluation per sample.
 Range (min … max):  8.744 ms …  10.707 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     9.138 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   9.108 ms ± 233.309 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

   ▁ ▃▄█▇▅▂    ▁▁ ▁    ▂ ▁▃▆▅▃▃▅▃▁▁  ▁                         
  ▆█▆██████▅▆▅▄██▇█▆▆▄▄█████████████▇█▇▇▇▄▄▄▃▃▄▁▄▃▃▃▃▄▁▃▁▁▁▁▃ ▅
  8.74 ms         Histogram: frequency by time        9.75 ms <

 Memory estimate: 2.83 KiB, allocs estimate: 88.

In [40]:
# speedup of GPU over CPU
median(bm_cpu.times) / median(bm_gpu.times)

3.9340775639184145

### Matrix multiplication

In [41]:
# SP matrix multiplication on GPU
bm_gpu = @benchmark Metal.@sync mul!($zd, $xd, $yd)

BenchmarkTools.Trial: 5 samples with 1 evaluation per sample.
 Range (min … max):  985.102 ms …   1.098 s  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):        1.072 s              ┊ GC (median):    0.00%
 Time  (mean ± σ):      1.061 s ± 44.733 ms  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █                                       █     █        █   █  
  █▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁█▁▁▁▁▁▁▁▁█▁▁▁█ ▁
  985 ms          Histogram: frequency by time           1.1 s <

 Memory estimate: 4.98 KiB, allocs estimate: 153.

For this problem size on this machine, we see GPU achieves a staggering **9 TFLOPS** throughput with single precision!

In [42]:
# SP throughput on GPU
(2n^3) / (minimum(bm_gpu.times) / 1e9)

8.92912055134507e12

In [43]:
# SP matrix multiplication on CPU
bm_cpu = @benchmark mul!($z, $x, $y)

BenchmarkTools.Trial: 1 sample with 1 evaluation per sample.
 Single result which took 12.849 s (0.00% GC) to evaluate,
 with a memory estimate of 0 bytes, over 0 allocations.

In [44]:
# SP throughput on CPU
(2n^3) / (minimum(bm_cpu.times) / 1e9)

6.845799704673441e11

In [45]:
# speedup on GPU over CPU
median(bm_cpu.times) / median(bm_gpu.times)

11.990426816269101

We see >10x speedup by GPUs in this matrix multiplication example.

### Cholesky decomposition

In [46]:
# cholesky on Gram matrix
# This one doesn't seem to work on Apple M2 chip yet
xtxd = xd'xd + I
@benchmark Metal.@sync cholesky($(xtxd))

ArgumentError: ArgumentError: cannot take the CPU address of a MtlMatrix{Float32, Metal.PrivateStorage}

In [47]:
xtx = collect(xtxd)
@benchmark LinearAlgebra.cholesky($(Symmetric(xtx)))

BenchmarkTools.Trial: 3 samples with 1 evaluation per sample.
 Range (min … max):  2.210 s …   2.250 s  ┊ GC (min … max): 0.04% … 0.06%
 Time  (median):     2.216 s              ┊ GC (median):    0.04%
 Time  (mean ± σ):   2.225 s ± 21.809 ms  ┊ GC (mean ± σ):  0.03% ± 0.03%

  █        █                                              █  
  █▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█ ▁
  2.21 s         Histogram: frequency by time        2.25 s <

 Memory estimate: 1.00 GiB, allocs estimate: 3.

## Evaluation of elementary and special functions on GPU

### Sine and log functions

In [48]:
# elementwise function on GPU arrays
fill!(yd, 1)
bm_gpu = @benchmark Metal.@sync $zd .= log.($yd .+ sin.($xd))
bm_gpu

BenchmarkTools.Trial: 552 samples with 1 evaluation per sample.
 Range (min … max):  8.755 ms …  10.040 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     9.110 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   9.067 ms ± 213.687 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

     ▄▂█▇▅             ▃▇▃▅▂    ▁                              
  ▅▅██████▅▇▆▄▅▆▄▄▄▄▄▄▄███████▅███▅▅▆▄▂▄▃▃▁▃▁▂▂▁▃▁▂▁▁▁▁▂▃▁▂▂▃ ▄
  8.75 ms         Histogram: frequency by time        9.73 ms <

 Memory estimate: 2.83 KiB, allocs estimate: 88.

In [49]:
# elementwise function on CPU arrays
x, y, z = collect(xd), collect(yd), collect(zd)
bm_cpu = @benchmark $z .= log.($y .+ sin.($x))
bm_cpu

BenchmarkTools.Trial: 2 samples with 1 evaluation per sample.
 Range (min … max):  2.702 s …  2.707 s  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     2.705 s             ┊ GC (median):    0.00%
 Time  (mean ± σ):   2.705 s ± 3.567 ms  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █                                                      █  
  █▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█ ▁
  2.7 s         Histogram: frequency by time        2.71 s <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [50]:
# Speed up of GPU over CPU for elementwise special functions
median(bm_cpu.times) / median(bm_gpu.times)

296.8859278495027

GPU brings great speedup (>50x) to the massive evaluation of elementary math functions.

### tanh function

In [51]:
bm_cpu = @benchmark z .= tanh.($x) # on CPU
bm_cpu

BenchmarkTools.Trial: 4 samples with 1 evaluation per sample.
 Range (min … max):  1.607 s …  1.609 s  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     1.607 s             ┊ GC (median):    0.00%
 Time  (mean ± σ):   1.608 s ± 1.119 ms  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █     █                       █                        █  
  █▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█ ▁
  1.61 s        Histogram: frequency by time        1.61 s <

 Memory estimate: 16 bytes, allocs estimate: 1.

In [52]:
bm_mtl = @benchmark zd .= Metal.@sync tanh.($xd) # Metal
bm_mtl

BenchmarkTools.Trial: 59 samples with 1 evaluation per sample.
 Range (min … max):   31.291 ms …    4.627 s  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):      32.364 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   124.796 ms ± 599.570 ms  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █                                                              
  █▄▆▄▁▁▇▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▄ ▁
  31.3 ms       Histogram: log(frequency) by time        463 ms <

 Memory estimate: 6.19 KiB, allocs estimate: 196.

Metal.jl accelerates the evaluation of tanh function by

In [53]:
median(bm_cpu.times) / median(bm_mtl.times)

49.66658748010938